# Descriptive Analysis of Parity and Its Determinants
## Sri Lanka Civil Registration and Vital Statistics (CRVS) Data
### Study Years: 2000 | 2005 | 2010 | 2015 | 2020

---

**Reporting Standards:**
- Birth weight reported in **grams (g)**, per WHO / SI convention
- Maternal age in **completed years**
- Proportions as **% to 1 decimal place**
- Means reported as **mean ± SD**; medians with **IQR [Q1, Q3]**
- All figures include axis labels with units, figure number, and caption
- Colour palette: colourblind-safe (seaborn `colorblind`)
- Chi-square significance threshold: **α = 0.05**; substantive threshold: **Cramér's V ≥ 0.10**

**Adaptive Design:** 2000 and 2005 datasets contain fewer variables than 2010–2020. Every analysis cell automatically detects which variables are present and skips gracefully when a variable is absent.

---

## 1. Library Imports

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import chi2_contingency
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display, Markdown
import warnings
import os

warnings.filterwarnings('ignore')

# ── Global style ─────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams.update({
    'font.family': 'Arial',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'figure.dpi': 150,
    'figure.figsize': (10, 5)
})

# Create output directories
os.makedirs('figures', exist_ok=True)
os.makedirs('tables', exist_ok=True)

print('All libraries loaded successfully.')

## 2. Global Configuration & Constants

All file paths, WHO/IUPAC thresholds, variable registry, and reporting constants are defined here. **Edit only this cell** when adapting to new data.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  FILE PATHS — update these to match your actual file locations
# ══════════════════════════════════════════════════════════════════════════════
DATA_PATHS = {
    2000: 'data/crvs_2000.csv',
    2005: 'data/crvs_2005.csv',
    2010: 'data/crvs_2010.csv',
    2015: 'data/crvs_2015.csv',
    2020: 'data/crvs_2020.csv',
}

STUDY_YEARS = [2000, 2005, 2010, 2015, 2020]

# ══════════════════════════════════════════════════════════════════════════════
#  WHO / IUPAC RANGE THRESHOLDS
# ══════════════════════════════════════════════════════════════════════════════

# Maternal age — valid range (completed years)
AGE_MIN, AGE_MAX = 10, 60           # outside → exclusion
AGE_FLAG_LOW  = 15                   # 10–14 → flagged adolescent
AGE_FLAG_HIGH = 49                   # 50–60 → flagged perimenopausal

# Birth weight — valid range (grams, SI unit)
BW_MIN, BW_MAX = 300, 6000          # outside → exclusion
BW_ELBW      = 1000                  # < 1000 g → ELBW
BW_VLBW      = 1500                  # < 1500 g → VLBW
BW_LBW       = 2500                  # < 2500 g → LBW (WHO threshold)
BW_NORMAL_HI = 4000                  # ≥ 4000 g → Macrosomia

# ══════════════════════════════════════════════════════════════════════════════
#  VARIABLE REGISTRY — which variables exist in which year
#  ✦ UPDATE THIS after inspecting each raw file's columns ✦
# ══════════════════════════════════════════════════════════════════════════════
VARIABLE_REGISTRY = {
    # Core variables — present in ALL years (2000–2020)
    'Birth_Order':           [2000, 2005, 2010, 2015, 2020],
    'Age_of_Mother':         [2000, 2005, 2010, 2015, 2020],
    'Marital_Status':        [2000, 2005, 2010, 2015, 2020],
    'Race_of_Mother':        [2000, 2005, 2010, 2015, 2020],
    'Gender':                [2000, 2005, 2010, 2015, 2020],
    'Hospital_or_Not':       [2000, 2005, 2010, 2015, 2020],
    'Multiple_Birth_Status': [2000, 2005, 2010, 2015, 2020],
    'Birth_Weight_g':        [2000, 2005, 2010, 2015, 2020],
    'District_of_Mother':    [2000, 2005, 2010, 2015, 2020],
    'Registered_District':   [2000, 2005, 2010, 2015, 2020],

    # Extended variables — only in 2010+
    'Race_of_Father':        [2010, 2015, 2020],
    'Gestational_Age_wk':    [2010, 2015, 2020],

    # Further extended — only in 2015+
    'Maternal_Education':    [2015, 2020],
    'Antenatal_Visits':      [2015, 2020],
}

# ══════════════════════════════════════════════════════════════════════════════
#  PARITY / REPORTING CONSTANTS
# ══════════════════════════════════════════════════════════════════════════════
PARITY_LABELS = {
    1: 'First',  2: 'Second', 3: 'Third',  4: 'Fourth',
    5: 'Fifth',  6: 'Sixth',  7: 'Seventh',8: 'Eighth',
    9: 'Ninth'
}

PARITY_MAP = {v: k for k, v in PARITY_LABELS.items()}

AGE_ORDER = ['<20', '20–24', '25–29', '30–34', '35+']

BW_CAT_ORDER = [
    'ELBW (<1000 g)', 'VLBW (1000–1499 g)', 'LBW (1500–2499 g)',
    'Normal (2500–3999 g)', 'Macrosomia (≥4000 g)'
]

print('Configuration loaded.')
print(f'Study years: {STUDY_YEARS}')
print(f'Registry variables: {len(VARIABLE_REGISTRY)}')

## 3. Helper Functions

In [ ]:
def has_var(df, varname):
    """Return True if varname is a column in df."""
    return varname in df.columns


def cramers_v(contingency_table):
    """Compute Cramér's V from a contingency table (pd.DataFrame or np.array)."""
    chi2 = chi2_contingency(contingency_table, correction=False)[0]
    n    = np.array(contingency_table).sum()
    r, k = contingency_table.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))


def bw_category(bw):
    """Assign WHO birth weight category label (units: grams)."""
    if pd.isna(bw):       return np.nan
    if   bw < BW_ELBW:    return 'ELBW (<1000 g)'
    elif bw < BW_VLBW:    return 'VLBW (1000–1499 g)'
    elif bw < BW_LBW:     return 'LBW (1500–2499 g)'
    elif bw < BW_NORMAL_HI: return 'Normal (2500–3999 g)'
    else:                  return 'Macrosomia (≥4000 g)'


def age_group(age):
    """Assign WHO-standard maternal age group (completed years)."""
    if pd.isna(age): return np.nan
    if   age < 20:   return '<20'
    elif age < 25:   return '20–24'
    elif age < 30:   return '25–29'
    elif age < 35:   return '30–34'
    else:            return '35+'


def fmt(n):
    """Format integer with thousand separators."""
    return f'{n:,}'


def section_header(title, year=None):
    """Print a clear section divider."""
    yr = f' — {year}' if year else ''
    print(f"\n{'═'*70}")
    print(f'  {title}{yr}')
    print(f"{'═'*70}")


print('Helper functions defined.')

## 4. Adaptive Data Loader

Loads any year's CSV, harmonises column names to canonical format, and reports which variables were detected vs. absent.

In [ ]:
def load_year(year):
    """
    Load one year's CRVS data, rename columns to canonical names,
    and detect which variables are available.
    """
    path = DATA_PATHS[year]
    df   = pd.read_csv(path, low_memory=False)

    # ── Column name harmonisation ────────────────────────────────────────
    # Add all known raw-file column name variants here
    RENAME_MAP = {
        # Birth order variants
        'Birth Order':        'Birth_Order',
        'birth_order':        'Birth_Order',
        'BirthOrder':         'Birth_Order',
        'BIRTH_ORDER':        'Birth_Order',
        # Maternal age variants
        'Age of Mother':      'Age_of_Mother',
        'age_mother':         'Age_of_Mother',
        'AgeOfMother':        'Age_of_Mother',
        'AGE_OF_MOTHER':      'Age_of_Mother',
        'Mother_Age':         'Age_of_Mother',
        # Marital status
        'Marital Status':     'Marital_Status',
        'marital_status':     'Marital_Status',
        # Race
        'Race of Mother':     'Race_of_Mother',
        'race_mother':        'Race_of_Mother',
        'Race of Father':     'Race_of_Father',
        'race_father':        'Race_of_Father',
        # Gender
        'Sex':                'Gender',
        'sex':                'Gender',
        'Child_Sex':          'Gender',
        # Hospital
        'Hospital or Not':    'Hospital_or_Not',
        'hospital_or_not':    'Hospital_or_Not',
        'Place_of_Delivery':  'Hospital_or_Not',
        # Multiple birth
        'Multiple Birth Status': 'Multiple_Birth_Status',
        'multiple_birth':     'Multiple_Birth_Status',
        # Birth weight
        'Birth Weight':       'Birth_Weight_g',
        'birth_weight':       'Birth_Weight_g',
        'BirthWeight':        'Birth_Weight_g',
        'Birth_Weight':       'Birth_Weight_g',
        'BIRTH_WEIGHT':       'Birth_Weight_g',
        # Districts
        'District of Mother': 'District_of_Mother',
        'district_mother':    'District_of_Mother',
        'Registered District':'Registered_District',
        'reg_district':       'Registered_District',
        # Extended variables (2010+)
        'Gestational Age':    'Gestational_Age_wk',
        'gest_age':           'Gestational_Age_wk',
        'Education':          'Maternal_Education',
        'Mother_Education':   'Maternal_Education',
        'Antenatal Visits':   'Antenatal_Visits',
        'antenatal_visits':   'Antenatal_Visits',
    }

    df.rename(columns=RENAME_MAP, inplace=True)

    # Strip whitespace from string columns
    str_cols = df.select_dtypes('object').columns
    for col in str_cols:
        df[col] = df[col].astype(str).str.strip()

    # Detect which registry variables are present
    detected = sorted([var for var in VARIABLE_REGISTRY
                       if var in df.columns])
    absent   = sorted([var for var in VARIABLE_REGISTRY
                       if var not in df.columns])

    section_header(f'DATA LOADED: {year}')
    print(f'  Raw records:       {fmt(len(df))}')
    print(f'  Raw columns:       {len(df.columns)}')
    print(f'  Registry detected: {len(detected)} / {len(VARIABLE_REGISTRY)}')
    print(f'  Present:  {detected}')
    print(f'  Absent:   {absent}')

    df.attrs['year']     = year
    df.attrs['detected'] = set(detected)
    return df


print('Data loader defined.')

---
# SECTION 1: DATA CLEANING
---

### 5. Birth Order Recoding
Convert text labels ('First' … 'Ninth') to numeric integers (1–9).

In [ ]:
def recode_birth_order(df):
    """
    Map text parity labels to integer 1–9.
    Handles multiple text formats (title case, upper, lower).
    """
    if not has_var(df, 'Birth_Order'):
        print('  ⚠ Birth_Order not found — skipping recoding')
        return df

    year = df.attrs.get('year', '?')

    # Normalise to title case first
    df['Birth_Order'] = df['Birth_Order'].astype(str).str.strip().str.title()

    label_map = {
        'First': 1, 'Second': 2, 'Third': 3, 'Fourth': 4,
        'Fifth': 5, 'Sixth': 6, 'Seventh': 7, 'Eighth': 8, 'Ninth': 9,
        '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9
    }

    df['Birth_Order_num'] = df['Birth_Order'].map(label_map)

    mapped   = df['Birth_Order_num'].notna().sum()
    unmapped = df['Birth_Order_num'].isna().sum()

    print(f'  [{year}] Birth_Order recoded: {fmt(mapped)} mapped, '
          f'{fmt(unmapped)} unmapped')

    if unmapped > 0:
        print(f'  Unmapped values: '
              f'{df.loc[df["Birth_Order_num"].isna(), "Birth_Order"].unique()[:10]}')

    return df


print('Birth order recoding function defined.')

### 6. Maternal Age Range Validation (Table 3.3 — WHO Standards)
Exclude ages < 10 and > 60 years. Flag 10–14 (adolescent) and 50–60 (perimenopausal).

In [ ]:
def validate_age(df):
    """
    Apply WHO-referenced age range validation per Table 3.3.
    Units: completed years.
    """
    if not has_var(df, 'Age_of_Mother'):
        print('  ⚠ Age_of_Mother not available — skipping age validation')
        return df

    year = df.attrs.get('year', '?')
    df['Age_of_Mother'] = pd.to_numeric(df['Age_of_Mother'], errors='coerce')

    # Exclusions
    n_excl_low  = (df['Age_of_Mother'] < AGE_MIN).sum()
    n_excl_high = (df['Age_of_Mother'] > AGE_MAX).sum()
    df['age_exclude'] = ((df['Age_of_Mother'] < AGE_MIN) |
                         (df['Age_of_Mother'] > AGE_MAX))

    # Flags per Table 3.3
    conditions = [
        df['Age_of_Mother'].between(10, 14),
        df['Age_of_Mother'].between(15, 49),
        df['Age_of_Mother'].between(50, 60),
    ]
    choices = ['Adolescent (10–14 yr)', 'WHO window (15–49 yr)',
              'Perimenopausal (50–60 yr)']
    df['age_flag'] = np.select(conditions, choices, default='Excluded / NaN')

    # Create age group variable
    df['Age_Group'] = df['Age_of_Mother'].apply(age_group)
    df['Age_Group'] = pd.Categorical(df['Age_Group'],
                                     categories=AGE_ORDER, ordered=True)

    print(f'  [{year}] Age validation:')
    print(f'    Excluded (< {AGE_MIN} yr):          {fmt(n_excl_low)}')
    print(f'    Excluded (> {AGE_MAX} yr):          {fmt(n_excl_high)}')
    print(f'    Flagged adolescent (10–14):  '
          f'{fmt((df["age_flag"] == "Adolescent (10–14 yr)").sum())}')
    print(f'    Flagged perimenopausal:      '
          f'{fmt((df["age_flag"] == "Perimenopausal (50–60 yr)").sum())}')
    return df


print('Age validation function defined.')

### 7. Birth Weight Range Validation (Table 3.4 — WHO Thresholds)
Exclude < 300 g and > 6,000 g. Assign WHO category labels. Flag VLBW and macrosomia.

In [ ]:
def validate_birthweight(df):
    """
    Apply WHO birth weight classification per Table 3.4.
    Units: grams (g) — SI unit per IUPAC convention.
    """
    if not has_var(df, 'Birth_Weight_g'):
        print('  ⚠ Birth_Weight_g not available — skipping BW validation')
        return df

    year = df.attrs.get('year', '?')
    df['Birth_Weight_g'] = pd.to_numeric(df['Birth_Weight_g'], errors='coerce')

    # Exclusions
    n_excl_low  = (df['Birth_Weight_g'] < BW_MIN).sum()
    n_excl_high = (df['Birth_Weight_g'] > BW_MAX).sum()
    df['bw_exclude'] = ((df['Birth_Weight_g'] < BW_MIN) |
                        (df['Birth_Weight_g'] > BW_MAX))

    # WHO category
    df['BW_Category'] = df['Birth_Weight_g'].apply(bw_category)
    df['BW_Category'] = pd.Categorical(df['BW_Category'],
                                       categories=BW_CAT_ORDER, ordered=True)

    # Flag extreme categories for sensitivity analysis
    df['bw_flag'] = ((df['Birth_Weight_g'] < BW_VLBW) |
                     (df['Birth_Weight_g'] >= BW_NORMAL_HI))

    print(f'  [{year}] Birth weight validation (units: grams):')
    print(f'    Excluded (< {BW_MIN} g):     {fmt(n_excl_low)}')
    print(f'    Excluded (> {BW_MAX} g):     {fmt(n_excl_high)}')
    print(f'    Flagged VLBW (<1500 g):   '
          f'{fmt((df["Birth_Weight_g"] < BW_VLBW).sum())}')
    print(f'    Flagged macrosomia (≥4000 g): '
          f'{fmt((df["Birth_Weight_g"] >= BW_NORMAL_HI).sum())}')

    # Summary distribution
    print(f'\n    WHO category distribution:')
    dist = df['BW_Category'].value_counts().sort_index()
    for cat, n in dist.items():
        pct = n / dist.sum() * 100
        print(f'      {cat}: {fmt(n)} ({pct:.1f}%)')
    return df


print('Birth weight validation function defined.')

### 8. Categorical Variable Standardisation
Harmonise text to title case; consolidate categories with < 30 records into 'Other'.

In [ ]:
def standardise_categoricals(df):
    """
    Harmonise categorical variables: title case, consolidate sparse categories.
    Only processes columns present in the dataframe.
    """
    year = df.attrs.get('year', '?')
    CAT_COLS = ['Marital_Status', 'Race_of_Mother', 'Race_of_Father',
                'Gender', 'Hospital_or_Not', 'Multiple_Birth_Status',
                'District_of_Mother', 'Registered_District',
                'Maternal_Education']

    print(f'  [{year}] Categorical standardisation:')
    for col in CAT_COLS:
        if not has_var(df, col):
            continue
        df[col] = df[col].astype(str).str.strip().str.title()
        # Replace 'Nan' strings from NaN conversion
        df[col] = df[col].replace({'Nan': np.nan, 'None': np.nan, '': np.nan})

        # Consolidate sparse categories
        counts = df[col].value_counts(dropna=True)
        sparse = counts[counts < 30].index.tolist()
        if sparse:
            df[col] = df[col].replace(sparse, 'Other')

        n_cats = df[col].nunique()
        print(f'    {col}: {n_cats} categories '
              f'({len(sparse)} sparse → Other)')
    return df


print('Categorical standardisation function defined.')

### 9. Structured Exclusion Log (Table 4.1 Template)
Apply all exclusions sequentially, recording N at every step.

In [ ]:
def apply_exclusions(df):
    """
    Apply sequential exclusions per methodology Section 3.4.
    Returns cleaned analytical dataframe and exclusion log.
    """
    year = df.attrs.get('year', '?')
    log  = []
    n0   = len(df)
    log.append(('Raw dataset (all registered live births)', '—', 0, n0))

    # Step 1: Age exclusions
    if 'age_exclude' in df.columns:
        n_before = len(df)
        df = df[~df['age_exclude']].copy()
        removed = n_before - len(df)
        log.append(('Age < 10 or > 60 yr (implausible)', 'Age_of_Mother',
                    removed, len(df)))

    # Step 2: Birth weight exclusions
    if 'bw_exclude' in df.columns:
        n_before = len(df)
        df = df[~df['bw_exclude']].copy()
        removed = n_before - len(df)
        log.append(('BW < 300 g or > 6000 g (implausible)', 'Birth_Weight_g',
                    removed, len(df)))

    # Steps 3–N: Listwise deletion per analytical variable
    LISTWISE_VARS = [
        'Birth_Order_num', 'Age_of_Mother', 'Marital_Status',
        'Race_of_Mother', 'Gender', 'Hospital_or_Not',
        'Multiple_Birth_Status', 'Birth_Weight_g'
    ]

    for var in LISTWISE_VARS:
        if not has_var(df, var):
            continue
        n_before = len(df)
        df = df[df[var].notna()].copy()
        removed = n_before - len(df)
        if removed > 0:
            log.append((f'Missing: {var}', var, removed, len(df)))

    log.append(('Final analytical sample', '—', 0, len(df)))

    # Print formatted log
    section_header('EXCLUSION LOG', year)
    print(f"{'Step':<50} {'Variable':<20} {'Removed':>10} {'N After':>10}")
    print('─' * 92)
    for step, var, removed, n_after in log:
        print(f'{step:<50} {var:<20} {fmt(removed):>10} {fmt(n_after):>10}')
    print('─' * 92)
    print(f'  Final N: {fmt(len(df))} '
          f'({len(df)/n0*100:.1f}% of raw records retained)')

    # Save log as CSV
    log_df = pd.DataFrame(log, columns=['Step','Variable','Removed','N_After'])
    log_df.to_csv(f'tables/exclusion_log_{year}.csv', index=False)

    df.attrs['year'] = year
    return df


print('Exclusion log function defined.')

### 10. Missing Data Analysis (Table 3.5 Equivalent)
Quantify missingness for every variable; classify Low / Moderate / High.

In [ ]:
def missing_analysis(df):
    """
    Compute missingness %, classify into tiers, print formatted table.
    Adaptive: only reports variables present in df.
    """
    year = df.attrs.get('year', '?')
    results = []

    # Only analyse registry variables that exist
    check_cols = [v for v in VARIABLE_REGISTRY if has_var(df, v)]

    for col in check_cols:
        n_miss = df[col].isna().sum()
        pct    = n_miss / len(df) * 100
        if pct <= 2:
            tier = 'Low (≤2%)'
        elif pct <= 10:
            tier = 'Moderate (>2–10%)'
        else:
            tier = 'High (>10%)'
        results.append({
            'Variable': col,
            'N Missing': n_miss,
            '% Missing': round(pct, 2),
            'Tier': tier
        })

    miss_df = pd.DataFrame(results)
    section_header('MISSING DATA ANALYSIS', year)
    print(miss_df.to_string(index=False))

    # Save
    miss_df.to_csv(f'tables/missingness_{year}.csv', index=False)
    return miss_df


print('Missing data analysis function defined.')

---
# SECTION 2: DESCRIPTIVE STATISTICS
---

### 11. Parity (Birth Order) Distribution — Table & Bar Chart

In [ ]:
def parity_distribution(df):
    """Frequency table + bar chart for birth order distribution."""
    if not has_var(df, 'Birth_Order_num'):
        print('  ⚠ Birth_Order_num not available'); return

    year   = df.attrs.get('year', '?')
    counts = df['Birth_Order_num'].value_counts().sort_index()
    pct    = counts / counts.sum() * 100
    cumpct = pct.cumsum()

    tbl = pd.DataFrame({
        'Birth Order': [PARITY_LABELS.get(i, str(i)) for i in counts.index],
        'n': counts.values,
        '% of Total': pct.round(1).values,
        'Cumulative %': cumpct.round(1).values
    })

    section_header('PARITY DISTRIBUTION', year)
    print(tbl.to_string(index=False))
    tbl.to_csv(f'tables/parity_distribution_{year}.csv', index=False)

    # ── Bar chart ────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = sns.color_palette('colorblind', len(counts))
    bars = ax.bar(counts.index, counts.values, color=colors, edgecolor='white')

    ax.set_xlabel('Birth Order (Parity)', fontsize=11)
    ax.set_ylabel('Number of Registered Live Births (n)', fontsize=11)
    ax.set_title(f'Figure: Distribution of Birth Order — '
                 f'Sri Lanka CRVS {year}', fontsize=13, fontweight='bold')
    ax.set_xticks(counts.index)
    ax.set_xticklabels([PARITY_LABELS.get(i, str(i)) for i in counts.index],
                        rotation=35, ha='right')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f'{x:,.0f}'))

    for bar, n in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + counts.max() * 0.015,
                f'{n:,}', ha='center', va='bottom', fontsize=8)

    plt.tight_layout()
    plt.savefig(f'figures/parity_dist_{year}.png', dpi=150, bbox_inches='tight')
    plt.show()


print('Parity distribution function defined.')

### 12. Maternal Age (yr) by Parity — Cross-tabulation & Boxplot

In [ ]:
def age_by_parity(df):
    """Cross-tabulation + boxplot of maternal age (completed years) by birth order."""
    if not has_var(df, 'Age_of_Mother') or not has_var(df, 'Birth_Order_num'):
        print('  ⚠ Required variables absent — skipping'); return

    year = df.attrs.get('year', '?')
    section_header('MATERNAL AGE × PARITY', year)

    # Cross-tabulation (% within each parity)
    ct = pd.crosstab(df['Age_Group'], df['Birth_Order_num'],
                     normalize='columns') * 100
    ct.columns = [PARITY_LABELS.get(c, str(c)) for c in ct.columns]
    print('\nAge Group Distribution Within Each Parity (column %):')
    print(ct.round(1).to_string())

    # Descriptive statistics per parity
    desc = df.groupby('Birth_Order_num')['Age_of_Mother'].agg(
        N='count',
        Mean='mean',
        SD='std',
        Median='median',
        Q1=lambda x: x.quantile(0.25),
        Q3=lambda x: x.quantile(0.75),
        Min='min',
        Max='max'
    ).round(1)
    desc.index = [PARITY_LABELS.get(i, str(i)) for i in desc.index]
    print(f'\nDescriptive Statistics: Maternal Age (years) by Parity')
    print(desc.to_string())
    desc.to_csv(f'tables/age_by_parity_{year}.csv')

    # ── Boxplot ──────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(11, 5))
    box_data = [df.loc[df['Birth_Order_num'] == p, 'Age_of_Mother'].dropna()
                for p in sorted(df['Birth_Order_num'].dropna().unique())]
    bp = ax.boxplot(box_data, patch_artist=True,
                    flierprops=dict(marker='.', markersize=2, alpha=0.3))

    colors = sns.color_palette('colorblind', len(box_data))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)

    parities = sorted(df['Birth_Order_num'].dropna().unique())
    ax.set_xticklabels([PARITY_LABELS.get(int(p), str(p)) for p in parities],
                        rotation=35, ha='right')
    ax.set_xlabel('Birth Order (Parity)', fontsize=11)
    ax.set_ylabel('Maternal Age (completed years)', fontsize=11)
    ax.set_title(f'Figure: Distribution of Maternal Age by Birth Order — '
                 f'Sri Lanka CRVS {year}', fontsize=12, fontweight='bold')

    # WHO window reference lines
    ax.axhline(15, ls='--', lw=0.8, color='orange', label='WHO lower bound (15 yr)')
    ax.axhline(49, ls='--', lw=0.8, color='red', label='WHO upper bound (49 yr)')
    ax.legend(fontsize=8, loc='upper left')

    plt.tight_layout()
    plt.savefig(f'figures/age_by_parity_{year}.png', dpi=150, bbox_inches='tight')
    plt.show()


print('Age by parity function defined.')

### 13. Race of Mother by Parity — Cross-tabulation & Stacked Bar

In [ ]:
def race_by_parity(df):
    """Cross-tabulation and stacked bar chart: Race of Mother × Birth Order."""
    if not has_var(df, 'Race_of_Mother') or not has_var(df, 'Birth_Order_num'):
        print('  ⚠ Required variables absent — skipping'); return

    year = df.attrs.get('year', '?')
    section_header('RACE OF MOTHER × PARITY', year)

    # Cross-tab: both absolute and column %
    ct_abs = pd.crosstab(df['Race_of_Mother'], df['Birth_Order_num'],
                         margins=True)
    ct_pct = pd.crosstab(df['Race_of_Mother'], df['Birth_Order_num'],
                         normalize='columns') * 100
    ct_pct.columns = [PARITY_LABELS.get(c, str(c)) for c in ct_pct.columns]

    print('\nRace of Mother Distribution Within Each Parity (column %):')
    print(ct_pct.round(1).to_string())
    ct_pct.to_csv(f'tables/race_by_parity_{year}.csv')

    # ── Stacked bar chart ────────────────────────────────────────────────
    ct_pct.T.plot(kind='bar', stacked=True, figsize=(11, 5),
                  colormap='tab10', edgecolor='white', linewidth=0.5)
    plt.xlabel('Birth Order (Parity)', fontsize=11)
    plt.ylabel('Proportion of Births (%)', fontsize=11)
    plt.title(f'Figure: Race of Mother by Birth Order — '
              f'Sri Lanka CRVS {year}', fontsize=12, fontweight='bold')
    plt.legend(title='Race of Mother', bbox_to_anchor=(1.02, 1),
               fontsize=9, title_fontsize=10)
    plt.xticks(rotation=35, ha='right')
    plt.tight_layout()
    plt.savefig(f'figures/race_by_parity_{year}.png', dpi=150, bbox_inches='tight')
    plt.show()


print('Race by parity function defined.')

### 14. Marital Status, Place of Delivery & Multiple Birth by Parity

In [ ]:
def categorical_by_parity(df):
    """Cross-tabulations for Marital Status, Hospital, Multiple Birth × Parity."""
    if not has_var(df, 'Birth_Order_num'):
        print('  ⚠ Birth_Order_num absent — skipping'); return

    year = df.attrs.get('year', '?')

    variables = [
        ('Marital_Status',        'Marital Status'),
        ('Hospital_or_Not',       'Place of Delivery'),
        ('Multiple_Birth_Status', 'Multiple Birth Status'),
        ('Gender',                'Child Sex'),
    ]

    for var, label in variables:
        if not has_var(df, var):
            print(f'  ⚠ {var} not available in {year} — skipping')
            continue

        section_header(f'{label.upper()} × PARITY', year)

        # Absolute counts
        ct_abs = pd.crosstab(df[var], df['Birth_Order_num'], margins=True)
        ct_abs.columns = [PARITY_LABELS.get(c, str(c)) for c in ct_abs.columns]

        # Column percentages
        ct_pct = pd.crosstab(df[var], df['Birth_Order_num'],
                             normalize='columns') * 100
        ct_pct.columns = [PARITY_LABELS.get(c, str(c)) for c in ct_pct.columns]

        print(f'\n{label} Distribution Within Each Parity (column %):')
        print(ct_pct.round(1).to_string())

        # Save
        ct_pct.to_csv(f'tables/{var}_by_parity_{year}.csv')


print('Categorical by parity function defined.')

### 15. Birth Weight (g) by Parity — The Classic Parity–Birth Weight Curve
Non-linear relationship: birth weight rises from parity 1 to 2–3, then declines (Ananth et al., 1995).

In [ ]:
def birthweight_by_parity(df):
    """Descriptive stats + boxplot + WHO category breakdown for BW by parity."""
    if not has_var(df, 'Birth_Weight_g') or not has_var(df, 'Birth_Order_num'):
        print('  ⚠ Required variables absent — skipping'); return

    year = df.attrs.get('year', '?')
    section_header('BIRTH WEIGHT (g) × PARITY', year)

    # Descriptive statistics
    desc = df.groupby('Birth_Order_num')['Birth_Weight_g'].agg(
        N='count',
        Mean_g='mean',
        SD_g='std',
        Median_g='median',
        Q1_g=lambda x: x.quantile(0.25),
        Q3_g=lambda x: x.quantile(0.75),
        LBW_pct=lambda x: (x < BW_LBW).mean() * 100
    ).round(1)
    desc.index = [PARITY_LABELS.get(i, str(i)) for i in desc.index]
    print('\nBirth Weight Descriptive Statistics by Parity:')
    print(f'  Units: grams (g) | LBW threshold: {BW_LBW} g (WHO, 2014)')
    print(desc.to_string())
    desc.to_csv(f'tables/bw_by_parity_{year}.csv')

    # WHO category prevalence per parity
    if has_var(df, 'BW_Category'):
        ct = pd.crosstab(df['BW_Category'], df['Birth_Order_num'],
                         normalize='columns') * 100
        ct.columns = [PARITY_LABELS.get(c, str(c)) for c in ct.columns]
        print(f'\nWHO Birth Weight Category (%) by Parity:')
        print(ct.round(1).to_string())

    # ── Boxplot with WHO reference lines ─────────────────────────────────
    fig, ax = plt.subplots(figsize=(11, 6))
    parities = sorted(df['Birth_Order_num'].dropna().unique())
    box_data = [df.loc[df['Birth_Order_num'] == p, 'Birth_Weight_g'].dropna()
                for p in parities]
    bp = ax.boxplot(box_data, patch_artist=True,
                    flierprops=dict(marker='.', markersize=1, alpha=0.2))

    colors = sns.color_palette('colorblind', len(box_data))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)

    ax.set_xticklabels([PARITY_LABELS.get(int(p), str(p)) for p in parities],
                        rotation=35, ha='right')

    # WHO reference lines
    for thresh, lbl, clr in [
        (BW_LBW,       f'LBW threshold ({BW_LBW} g)',       'orange'),
        (BW_ELBW,      f'ELBW threshold ({BW_ELBW} g)',     'red'),
        (BW_NORMAL_HI, f'Macrosomia threshold ({BW_NORMAL_HI} g)', 'purple'),
    ]:
        ax.axhline(thresh, ls='--', lw=0.9, color=clr, label=lbl)

    ax.set_xlabel('Birth Order (Parity)', fontsize=11)
    ax.set_ylabel('Birth Weight (g)', fontsize=11)
    ax.set_title(f'Figure: Birth Weight (g) by Birth Order — '
                 f'Sri Lanka CRVS {year}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=8, loc='upper right')
    plt.tight_layout()
    plt.savefig(f'figures/bw_by_parity_{year}.png', dpi=150, bbox_inches='tight')
    plt.show()


print('Birth weight by parity function defined.')

### 16. Geographic Distribution by District

In [ ]:
def geographic_distribution(df):
    """Distribution of births by district; highlights conflict-area underrepresentation."""
    year = df.attrs.get('year', '?')

    for geo_col in ['District_of_Mother', 'Registered_District']:
        if not has_var(df, geo_col):
            continue

        section_header(f'{geo_col.upper()} DISTRIBUTION', year)

        ct = df[geo_col].value_counts().reset_index()
        ct.columns = ['District', 'n']
        ct['%'] = (ct['n'] / ct['n'].sum() * 100).round(1)
        ct['Cumulative %'] = ct['%'].cumsum().round(1)
        print(ct.to_string(index=False))
        ct.to_csv(f'tables/{geo_col}_{year}.csv', index=False)

        # Bar chart — top 15 districts
        top15 = ct.head(15)
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.barh(top15['District'][::-1], top15['n'][::-1],
                color=sns.color_palette('colorblind', 15))
        ax.set_xlabel('Number of Registered Live Births (n)', fontsize=11)
        ax.set_title(f'Figure: Top 15 Districts by {geo_col.replace("_", " ")} — '
                     f'Sri Lanka CRVS {year}', fontsize=12, fontweight='bold')
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(
            lambda x, _: f'{x:,.0f}'))
        plt.tight_layout()
        plt.savefig(f'figures/{geo_col}_{year}.png', dpi=150, bbox_inches='tight')
        plt.show()


print('Geographic distribution function defined.')

### 17. Extended Variables — Available Only in 2010+ and 2015+
These blocks automatically skip for 2000 and 2005.

In [ ]:
def extended_variables_by_parity(df):
    """
    Analyse variables only available in later years:
    - Race_of_Father (2010+)
    - Gestational_Age_wk (2010+)
    - Maternal_Education (2015+)
    - Antenatal_Visits (2015+)
    """
    if not has_var(df, 'Birth_Order_num'):
        return

    year = df.attrs.get('year', '?')

    # ── Race of Father (2010+) ───────────────────────────────────────────
    if has_var(df, 'Race_of_Father'):
        section_header('RACE OF FATHER × PARITY', year)
        ct = pd.crosstab(df['Race_of_Father'], df['Birth_Order_num'],
                         normalize='columns') * 100
        ct.columns = [PARITY_LABELS.get(c, str(c)) for c in ct.columns]
        print(ct.round(1).to_string())
        ct.to_csv(f'tables/race_father_by_parity_{year}.csv')
    else:
        print(f'  ⚠ Race_of_Father not available in {year} — skipping')

    # ── Gestational Age (2010+) ──────────────────────────────────────────
    if has_var(df, 'Gestational_Age_wk'):
        section_header('GESTATIONAL AGE (weeks) × PARITY', year)
        df['Gestational_Age_wk'] = pd.to_numeric(
            df['Gestational_Age_wk'], errors='coerce')
        desc = df.groupby('Birth_Order_num')['Gestational_Age_wk'].agg(
            N='count', Mean_wk='mean', SD_wk='std',
            Median_wk='median',
            Q1_wk=lambda x: x.quantile(0.25),
            Q3_wk=lambda x: x.quantile(0.75),
            Preterm_pct=lambda x: (x < 37).mean() * 100
        ).round(1)
        desc.index = [PARITY_LABELS.get(i, str(i)) for i in desc.index]
        print('  Units: completed weeks')
        print(desc.to_string())
        desc.to_csv(f'tables/gest_age_by_parity_{year}.csv')
    else:
        print(f'  ⚠ Gestational_Age_wk not available in {year} — skipping')

    # ── Maternal Education (2015+) ───────────────────────────────────────
    if has_var(df, 'Maternal_Education'):
        section_header('MATERNAL EDUCATION × PARITY', year)
        ct = pd.crosstab(df['Maternal_Education'], df['Birth_Order_num'],
                         normalize='columns') * 100
        ct.columns = [PARITY_LABELS.get(c, str(c)) for c in ct.columns]
        print(ct.round(1).to_string())
        ct.to_csv(f'tables/education_by_parity_{year}.csv')
    else:
        print(f'  ⚠ Maternal_Education not available in {year} — skipping')

    # ── Antenatal Visits (2015+) ─────────────────────────────────────────
    if has_var(df, 'Antenatal_Visits'):
        section_header('ANTENATAL VISITS × PARITY', year)
        df['Antenatal_Visits'] = pd.to_numeric(
            df['Antenatal_Visits'], errors='coerce')
        desc = df.groupby('Birth_Order_num')['Antenatal_Visits'].agg(
            N='count', Mean='mean', SD='std',
            Median='median'
        ).round(1)
        desc.index = [PARITY_LABELS.get(i, str(i)) for i in desc.index]
        print(desc.to_string())
        desc.to_csv(f'tables/antenatal_by_parity_{year}.csv')
    else:
        print(f'  ⚠ Antenatal_Visits not available in {year} — skipping')


print('Extended variables function defined.')

---
# SECTION 3: BIVARIATE ANALYSIS — CHI-SQUARE & CRAMÉR'S V
---

In [ ]:
def chisq_summary(df):
    """
    Chi-square tests of independence: each predictor × Birth_Order.
    Reports χ², df, p-value, Cramér's V, and significance assessment.
    Threshold: α = 0.05 for significance; V ≥ 0.10 for substantive.
    """
    if not has_var(df, 'Birth_Order_num'):
        print('  ⚠ Birth_Order_num absent — skipping'); return

    year = df.attrs.get('year', '?')
    section_header('CHI-SQUARE BIVARIATE ANALYSIS', year)

    PREDICTORS = [
        # Core variables (all years)
        'Age_Group', 'Race_of_Mother', 'Marital_Status',
        'Gender', 'Hospital_or_Not', 'Multiple_Birth_Status',
        'BW_Category',
        # Extended variables (later years)
        'Race_of_Father', 'Maternal_Education',
    ]

    rows = []
    for pred in PREDICTORS:
        if not has_var(df, pred):
            continue

        # Drop NaN for the contingency table
        sub = df[[pred, 'Birth_Order_num']].dropna()
        if sub.empty or sub[pred].nunique() < 2:
            continue

        ct = pd.crosstab(sub[pred], sub['Birth_Order_num'])
        chi2, p, dof, expected = chi2_contingency(ct, correction=False)
        v = cramers_v(ct)

        sig   = 'Yes' if p < 0.05 else 'No'
        subst = 'Yes' if v >= 0.10 else 'No'

        rows.append({
            'Predictor': pred,
            'χ²': round(chi2, 2),
            'df': dof,
            'p-value': f'{p:.4e}' if p < 0.0001 else f'{p:.4f}',
            'Sig (α=0.05)': sig,
            "Cramér's V": round(v, 4),
            'Substantive (V≥0.10)': subst
        })

    result_df = pd.DataFrame(rows)
    print(result_df.to_string(index=False))
    result_df.to_csv(f'tables/chisq_summary_{year}.csv', index=False)
    return result_df


print('Chi-square summary function defined.')

### VIF — Variance Inflation Factor
Assess multicollinearity before regression modelling. VIF > 5 = moderate; VIF > 10 = problematic.

In [ ]:
def compute_vif(df):
    """Compute VIF for all dummy-encoded predictors."""
    from statsmodels.stats.outliers_influence import variance_inflation_factor

    year = df.attrs.get('year', '?')
    section_header('VARIANCE INFLATION FACTOR (VIF)', year)

    # Select predictors that exist
    cat_vars = [v for v in ['Race_of_Mother', 'Marital_Status',
                            'Gender', 'Hospital_or_Not',
                            'Multiple_Birth_Status', 'Age_Group',
                            'Maternal_Education']
                if has_var(df, v)]
    cont_vars = [v for v in ['Birth_Weight_g', 'Age_of_Mother',
                             'Gestational_Age_wk', 'Antenatal_Visits']
                 if has_var(df, v)]

    sub = df[cat_vars + cont_vars].dropna()
    if sub.empty:
        print('  No complete cases for VIF — skipping'); return

    # Dummy encode categoricals
    encoded = pd.get_dummies(sub[cat_vars], drop_first=True, dtype=float)
    for v in cont_vars:
        encoded[v] = sub[v].astype(float)

    # Add constant
    encoded.insert(0, 'const', 1.0)

    vif_data = []
    for i in range(1, encoded.shape[1]):  # skip constant
        vif_val = variance_inflation_factor(encoded.values, i)
        flag = '⚠ MODERATE' if vif_val > 5 else ('🚨 HIGH' if vif_val > 10 else '')
        vif_data.append({
            'Variable': encoded.columns[i],
            'VIF': round(vif_val, 2),
            'Flag': flag
        })

    vif_df = pd.DataFrame(vif_data).sort_values('VIF', ascending=False)
    print(vif_df.to_string(index=False))
    vif_df.to_csv(f'tables/vif_{year}.csv', index=False)
    return vif_df


print('VIF function defined.')

---
# SECTION 4: CROSS-YEAR COMPARATIVE ANALYSIS (2000–2020)
---

### 18. Multi-Year Processing Pipeline
Loads, cleans, and analyses all five years in a single loop.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  MASTER PIPELINE — Process all years
# ══════════════════════════════════════════════════════════════════════════════

results = {}  # stores cleaned analytical dataframes for all years

for year in STUDY_YEARS:
    print(f'\n{"#" * 70}')
    print(f'  PROCESSING YEAR: {year}')
    print(f'{"#" * 70}')

    try:
        # ── Load ─────────────────────────────────────────────────────────
        df = load_year(year)

        # ── Clean ────────────────────────────────────────────────────────
        df = recode_birth_order(df)
        df = validate_age(df)
        df = validate_birthweight(df)
        df = standardise_categoricals(df)

        # ── Missing data analysis (before exclusions) ────────────────────
        missing_analysis(df)

        # ── Apply exclusions ─────────────────────────────────────────────
        df = apply_exclusions(df)

        # ── Store ────────────────────────────────────────────────────────
        results[year] = df
        print(f'\n  ✓ Year {year} processed successfully: '
              f'{fmt(len(df))} analytical records')

    except FileNotFoundError:
        print(f'  ⚠ File not found for {year}: {DATA_PATHS[year]}')
        print(f'    Skipping this year. Update DATA_PATHS in Block 2.')
    except Exception as e:
        print(f'  ❌ Error processing {year}: {e}')
        import traceback; traceback.print_exc()

print(f'\n{"═" * 70}')
print(f'  Years successfully processed: {sorted(results.keys())}')
print(f'{"═" * 70}')

### 19. Run All Descriptive Analyses Per Year

In [ ]:
for year, df in results.items():
    print(f'\n{"#" * 70}')
    print(f'  DESCRIPTIVE ANALYSIS — {year}')
    print(f'{"#" * 70}')

    # Core descriptives (all years)
    parity_distribution(df)
    age_by_parity(df)
    race_by_parity(df)
    categorical_by_parity(df)
    birthweight_by_parity(df)
    geographic_distribution(df)

    # Extended descriptives (adaptive — skips if variables absent)
    extended_variables_by_parity(df)

    # Bivariate tests
    chisq_summary(df)
    compute_vif(df)

print('\n✓ All descriptive analyses complete.')

### 20. Cross-Year Trend Summary Table

In [ ]:
trend_rows = []

for year, df in sorted(results.items()):
    row = {'Year': year, 'N (analytical)': len(df)}

    if has_var(df, 'Birth_Order_num'):
        row['Mean Parity']   = round(df['Birth_Order_num'].mean(), 2)
        row['Median Parity'] = df['Birth_Order_num'].median()
        row['% First Birth'] = round(
            (df['Birth_Order_num'] == 1).mean() * 100, 1)
        row['% Parity ≥4']   = round(
            (df['Birth_Order_num'] >= 4).mean() * 100, 1)

    if has_var(df, 'Birth_Weight_g'):
        row['Mean BW (g)']     = round(df['Birth_Weight_g'].mean(), 1)
        row['% LBW (<2500 g)'] = round(
            (df['Birth_Weight_g'] < BW_LBW).mean() * 100, 1)

    if has_var(df, 'Age_of_Mother'):
        row['Mean Maternal Age (yr)'] = round(
            df['Age_of_Mother'].mean(), 1)

    if has_var(df, 'Hospital_or_Not'):
        hosp_vals = df['Hospital_or_Not'].str.lower()
        row['% Hospital Delivery'] = round(
            hosp_vals.isin(['hospital', 'yes']).mean() * 100, 1)

    trend_rows.append(row)

trend_df = pd.DataFrame(trend_rows).set_index('Year')

section_header('CROSS-YEAR TREND SUMMARY — 2000 to 2020')
print(trend_df.to_string())
trend_df.to_csv('tables/trend_summary_2000_2020.csv')
print('\n  Saved to tables/trend_summary_2000_2020.csv')

### 21. Cross-Year Trend Visualisations

In [ ]:
if len(results) < 2:
    print('  ⚠ Need at least 2 years for trend plots — skipping')
else:
    years = sorted(results.keys())
    n_plots = 4
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    # Plot 1: Mean Parity
    if 'Mean Parity' in trend_df.columns:
        axes[0].plot(trend_df.index, trend_df['Mean Parity'],
                     marker='o', linewidth=2, color='#2E5FA3')
        axes[0].set_title('Mean Birth Order (Parity)', fontweight='bold')
        axes[0].set_ylabel('Mean Parity')
        axes[0].grid(True, alpha=0.3)

    # Plot 2: % First Births
    if '% First Birth' in trend_df.columns:
        axes[1].plot(trend_df.index, trend_df['% First Birth'],
                     marker='s', linewidth=2, color='#2E8B57')
        axes[1].set_title('% First Births', fontweight='bold')
        axes[1].set_ylabel('% of Total Births')
        axes[1].grid(True, alpha=0.3)

    # Plot 3: LBW Prevalence
    if '% LBW (<2500 g)' in trend_df.columns:
        axes[2].plot(trend_df.index, trend_df['% LBW (<2500 g)'],
                     marker='^', linewidth=2, color='#D35400')
        axes[2].axhline(16, ls='--', lw=1, color='grey',
                        label='WHO benchmark (~16%)')
        axes[2].set_title('LBW Prevalence (< 2500 g)', fontweight='bold')
        axes[2].set_ylabel('% LBW')
        axes[2].legend(fontsize=9)
        axes[2].grid(True, alpha=0.3)

    # Plot 4: Mean Maternal Age
    if 'Mean Maternal Age (yr)' in trend_df.columns:
        axes[3].plot(trend_df.index, trend_df['Mean Maternal Age (yr)'],
                     marker='D', linewidth=2, color='#8E44AD')
        axes[3].set_title('Mean Maternal Age (yr)', fontweight='bold')
        axes[3].set_ylabel('Mean Age (years)')
        axes[3].grid(True, alpha=0.3)

    for ax in axes:
        ax.set_xlabel('Year')
        ax.set_xticks(years)

    plt.suptitle('Trends in Parity and Birth Outcomes — '
                 'Sri Lanka CRVS 2000–2020',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('figures/trend_overview_2000_2020.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    print('  Saved to figures/trend_overview_2000_2020.png')

### 22. Parity Distribution Comparison Across All Years

In [ ]:
if len(results) >= 2:
    fig, ax = plt.subplots(figsize=(12, 6))
    width = 0.15
    years_sorted = sorted(results.keys())

    for i, year in enumerate(years_sorted):
        df = results[year]
        if not has_var(df, 'Birth_Order_num'):
            continue
        counts = df['Birth_Order_num'].value_counts().sort_index()
        pct    = counts / counts.sum() * 100
        x      = np.arange(len(pct))
        ax.bar(x + i * width, pct.values, width=width,
               label=str(year), alpha=0.85)

    ax.set_xlabel('Birth Order (Parity)', fontsize=11)
    ax.set_ylabel('% of Total Births', fontsize=11)
    ax.set_title('Comparison of Parity Distribution — '
                 'Sri Lanka CRVS 2000–2020',
                 fontsize=13, fontweight='bold')
    ax.set_xticks(x + width * len(years_sorted) / 2)
    ax.set_xticklabels([PARITY_LABELS.get(int(p), str(p))
                        for p in sorted(pct.index)], rotation=35, ha='right')
    ax.legend(title='Year', fontsize=10)
    plt.tight_layout()
    plt.savefig('figures/parity_comparison_all_years.png',
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('  ⚠ Need at least 2 years for comparison plot')

---
# SECTION 5: NOTEBOOK SUMMARY
---

In [ ]:
section_header('NOTEBOOK EXECUTION SUMMARY')
print(f'  Years processed: {sorted(results.keys())}')
print(f'  Total analytical records across all years: '
      f'{fmt(sum(len(df) for df in results.values()))}')
print()

for year in sorted(results.keys()):
    df = results[year]
    detected = df.attrs.get('detected', set())
    absent   = set(VARIABLE_REGISTRY.keys()) - detected
    print(f'  {year}: N = {fmt(len(df)):>10} | '
          f'{len(detected)} vars detected | '
          f'{len(absent)} vars absent')
    if absent:
        print(f'         Absent: {sorted(absent)}')

print(f'\n  Output files saved to:')
print(f'    - tables/  (CSV files for all cross-tabulations)')
print(f'    - figures/  (PNG figures for all plots)')
print(f'\n  ✓ Notebook complete.')